# 4. Non-Stationary Transformer Playground

In this notebook, we execute the refined, modularized version of the **Non-Stationary Transformer** (NeurIPS 2022). 

Following our MLOps standards, the internal PyTorch mathematics (Series Stationarization and De-stationary Attention) are encapsulated in `src/models/transformer_model.py`. This notebook serves as an interactive entry point to trigger the full pipeline: **Preprocessing -> Cluster Training -> Rolling Forecast -> Evaluation**.

### Pipeline Overview:
1. **Device Detection**: Optimized execution on MPS (Mac), CUDA (Nvidia), or CPU.
2. **Cluster Aggregation**: Training one neural model per behavioral shape (Light, Medium, Heavy, etc.) using hourly averages.
3. **Rolling Window Forecast**: Simulating a day-ahead operational scenario.
4. **Safe Merge 2.0**: Resampling cluster forecasts back to 15-minute intervals and un-scaling via individual client scalers.

In [ ]:
import os
import sys
import torch
import pandas as pd

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.transformer_model import run_transformer_pipeline
from src.tools.visualization import plot_cluster_portfolio, analyze_time_periods

# ---------------------------------------------------------
# Device selection: MPS (Apple Silicon) > CUDA > CPU
# ---------------------------------------------------------
device = torch.device("mps" if torch.backends.mps.is_available() 
                      else "cuda" if torch.cuda.is_available() 
                      else "cpu")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Active Device: {device.type.upper()}")

## 1. Run Modular Pipeline

We run the `day_ahead` scenario. The script will:
- Load the processed Parquet dataset.
- Segment and scale consumption at the client level.
- Train 5 neural models (one per cluster).
- Compute WMAPE/MAPE metrics for the entire portfolio.

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, "Datasets", "processed_electricity_data.parquet")

cluster_models, test_raw, portfolio_eval, summary = run_transformer_pipeline(
    DATA_PATH, 
    mode='day_ahead', 
    plot=True
)

## 2. Statistical Deep Dive

Beyond the cluster-level portfolio, we analyze the Transformer's performance across different temporal slices (Weekdays vs Weekends, Peak vs Off-Peak hours).

In [ ]:
analyze_time_periods(test_raw)